In [1]:
import os
import torch
import torch.nn as nn
from transformers import GPT2Tokenizer
from datasets import load_dataset

# Import custom scripts
from data_utils import ContiguousDataLoader
from trainer import train_universal_model
from evals_utils import plot_training_metrics

# models
from models.FLB_Model import FLB_Transformer 

# Optimize PyTorch for your hardware
torch.backends.cudnn.benchmark = True
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: cuda


## Hyperparameter Configuration

In [ ]:
# We use GPT-2's vocabulary size since we are using its tokenizer
VOCAB_SIZE = 50257

# Model Dimensions
HIDDEN_DIM = 256
NUM_LAYERS = 6
NUM_HEADS = 8

# TBPTT & Sequence Length
SEQ_LEN = 128
BATCH_SIZE = 48
ACCUMULATION_STEPS = 1 # Effective batch size = 128 * 2 = 256

# Training Parameters
LEARNING_RATE = 3e-4
MAX_NORM = 5.0 # Gradient clipping norm
EPOCHS = 10
LOG_INTERVAL = 10

# Define where you want to save the processed tensor
TOKEN_CACHE_PATH = 'wikitext2_tokens.pt'

## Data Ingestion (WikiText-2)

In [3]:
if os.path.exists(TOKEN_CACHE_PATH):
    print(f"Found cached tokens at {TOKEN_CACHE_PATH}. Loading directly into memory...")
    tokens = torch.load(TOKEN_CACHE_PATH).to('cuda') 
    print(f"Total tokens in dataset: {len(tokens):,}")
    
else:
    print("No cache found. Downloading WikiText-2 and GPT-2 Tokenizer...")
    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    dataset = load_dataset('wikitext', 'wikitext-2-raw-v1', split='train')

    # Filter out empty lines and join the text into one massive string
    text_data = "\n".join([text for text in dataset['text'] if text.strip() != ""])

    # Tokenize the entire corpus into a single 1D array
    print("Tokenizing corpus (this might take a minute)...")
    tokens = tokenizer.encode(text_data, return_tensors='pt').squeeze(0)
    print(f"Total tokens in dataset: {len(tokens):,}")
    
    # Save the 1D tensor to disk so we never have to do this again
    print(f"Saving tokenized tensor to {TOKEN_CACHE_PATH}...")
    torch.save(tokens, TOKEN_CACHE_PATH)

# Initialize your custom contiguous data loader
dataloader = ContiguousDataLoader(tokens, batch_size=BATCH_SIZE, seq_len=SEQ_LEN)
print(f"Created {len(dataloader)} micro-batches per epoch.")

Found cached tokens at wikitext2_tokens.pt. Loading directly into memory...


/tmp/ipykernel_67867/238101887.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  tokens = torch.load(TOKEN_CACHE_PATH).to('cuda')


Total tokens in dataset: 2,391,884
Created 389 micro-batches per epoch.


## Model Instantiation

In [4]:
model = FLB_Transformer(
    vocab_size=VOCAB_SIZE, 
    hidden_dim=HIDDEN_DIM, 
    num_layers=NUM_LAYERS, 
    seq_len=SEQ_LEN, 
    batch_size=BATCH_SIZE,
    dropout=0.1, 
    aux_weight=1.0
).to(device)

print("Compiling the model...")
torch.compile(model)

# Count parameters to ensure fair benchmarking later
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"FLB-Transformer initialized with {total_params:,} trainable parameters.")

Compiling the model...
FLB-Transformer initialized with 28,938,321 trainable parameters.


## Training Execution

In [5]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

print("Starting training...")
# This will safely log to training_log.csv without bottlenecking your GPU
trained_model = train_universal_model(
    model=model, 
    dataloader=dataloader, 
    optimizer=optimizer, 
    epochs=EPOCHS, 
    accumulation_steps=ACCUMULATION_STEPS, 
    log_interval=LOG_INTERVAL,
    max_norm=MAX_NORM,
    device=device,
    log_file='artifacts/logs/flb_training_log.csv',
    save_dir='artifacts/checkpoints'
)

Starting training...
--- Found existing checkpoint: artifacts/checkpoints/model_ep0_complete.pt ---


/home/abrah/Projects/FLB-Transformer/training/trainer.py:41: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(latest_checkpoint, map_location=device)


--- Successfully resumed from Epoch 1 ---
--- Loaded 38 existing telemetry entries ---


Epoch 2/10:   0%|          | 0/389 [00:00<?, ?it/s]

KeyboardInterrupt: 

## Artifact Generation

In [ ]:
print("Generating thesis artifacts...")
# FIX: Use the full path where the logs are actually saved
plot_training_metrics(
    log_file='artifacts/logs/flb_training_log.csv', 
    output_dir='flb_plots'
)

# Display the plot
from IPython.display import Image, display
display(Image(filename='artifacts/plots/loss_curve.png'))

Generating thesis artifacts...


FileNotFoundError: [Errno 2] No such file or directory: 'flb_training_log.csv'